# v1.3 Dataset Builder: User + Test + Cohort

Purpose: build the v1.3 Test / Exercise Readiness dataset from database tables without embedding credentials in the notebook.

This notebook is intentionally not the full Learn Smarter model. It produces attempt-level data, DQ flags, a published KPI dataset, and a proxy-sequence dataset for BLS/ALS/CAS proxy analysis.

Sensitive connection details should be added locally by the operator and should not be committed.

## v1.3 Boundaries

- Topic-level readiness is a v2 target because the current source does not yet expose canonical `topic_id` / `topic_name` fields.
- Subject-level context can be inferred cautiously from `tests.name`, but it is not canonical subject metadata.
- `class_id` can support cohort analysis if joined through learner/subscriber/class membership and attempt timestamps.
- BLS, ALS, CAS, and learning gain are proxy signals only in v1.3.
- DQ must explain exclusions. Do not filter rows so early that the audit trail is lost.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

try:
    from sqlalchemy import create_engine, inspect, text
except ImportError:
    create_engine = None
    inspect = None
    text = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

## Configuration

Fill in the connection locally. Recommended options:

- use Colab secrets
- use environment variables
- paste a SQLAlchemy URL only in your private notebook copy

Do not commit secrets back to GitHub.

In [ ]:
PROJECT_ROOT = Path.cwd()
EXPORT_DIR = PROJECT_ROOT / "data" / "v1_3_dataset_build"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TABLES = {
    "test_takers": "test_takers",
    "test_results": "test_results",
    "test_questions": "test_questions",
    "test_answers": "test_answers",
    "tests": "tests",
    "users": "users",
    "test_classes": "test_classes",
    # Replace this once the real learner/subscriber/class membership table is confirmed.
    "class_memberships": None,
}

DATE_START = None  # Example: "2025-01-01"
DATE_END = None    # Example: "2026-01-01"

RUN_LABEL = "v1_3_user_test_cohort_build"

In [ ]:
def build_engine():
    """Return a SQLAlchemy engine. Fill this in locally; do not commit credentials."""
    if create_engine is None:
        raise ImportError("Install sqlalchemy and the database driver for your connection.")

    # Option A: environment variable
    # import os
    # url = os.environ["ECAMPUS_DB_URL"]
    # return create_engine(url)

    # Option B: Google Colab secrets
    # from google.colab import userdata
    # url = userdata.get("ECAMPUS_DB_URL")
    # return create_engine(url)

    raise NotImplementedError("Add private DB connection details in your local copy.")

## Schema Inspection

Run this before writing final SQL. The older notebook used CSV exports, so this step confirms what the live database actually exposes.

In [ ]:
def inspect_tables(engine, table_names):
    db = inspect(engine)
    rows = []
    for logical_name, table_name in table_names.items():
        if not table_name:
            rows.append({"logical_name": logical_name, "table_name": None, "column_name": None, "type": None})
            continue
        try:
            columns = db.get_columns(table_name)
        except Exception as exc:
            rows.append({"logical_name": logical_name, "table_name": table_name, "column_name": "<inspect_failed>", "type": str(exc)})
            continue
        for column in columns:
            rows.append({
                "logical_name": logical_name,
                "table_name": table_name,
                "column_name": column.get("name"),
                "type": str(column.get("type")),
            })
    return pd.DataFrame(rows)

# engine = build_engine()
# schema_df = inspect_tables(engine, TABLES)
# display(schema_df)
# schema_df.to_csv(EXPORT_DIR / "schema_inventory.csv", index=False)

## Raw Extraction Query

Goal: create the widest useful attempt-level candidate dataset before DQ filtering.

Keep raw rows and derived DQ flags. Do not only export the filtered dataset.

`marks` is sourced from `test_takers.marks` because answer/result row grade sums may include both correct and wrong answer rows. The answer-rollup grade sum is kept only as a diagnostic field.

In [ ]:
RAW_ATTEMPT_SQL = """
-- Adjust column names after schema inspection.
-- This query should return one row per test_taker attempt.
WITH answer_rollup AS (
    SELECT
        tr.test_taker_id,
        COUNT(*) AS answer_rows,
        COUNT(tr.chosen_answer_id) AS attempted_questions,
        SUM(CASE WHEN ta.correct = 1 THEN 1 ELSE 0 END) AS correct_answers,
        SUM(CASE WHEN tr.chosen_answer_id IS NOT NULL AND COALESCE(ta.correct, 0) = 0 THEN 1 ELSE 0 END) AS wrong_answers,
        SUM(COALESCE(tr.grade, 0)) AS answer_grade_sum_diagnostic
    FROM {test_results} tr
    LEFT JOIN {test_answers} ta
        ON ta.answer_id = tr.chosen_answer_id
    GROUP BY tr.test_taker_id
), question_rollup AS (
    SELECT
        tq.test_id,
        COUNT(*) AS total_questions,
        SUM(COALESCE(tq.grade, 1)) AS total_marks
    FROM {test_questions} tq
    WHERE COALESCE(tq.is_active, 1) = 1
    GROUP BY tq.test_id
), test_class_rollup AS (
    SELECT
        tc.test_id,
        MIN(tc.class_id) AS class_id,
        MIN(tc.group_id) AS group_id,
        COUNT(DISTINCT tc.class_id) AS test_class_count
    FROM {test_classes} tc
    WHERE COALESCE(tc.active, 1) = 1
    GROUP BY tc.test_id
)
SELECT
    tt.test_taker_id AS attempt_id,
    tt.test_taker_id,
    tt.user_id,
    tt.subscriber_id,
    u.f_name AS first_name,
    u.l_name AS last_name,
    u.username AS learner_id,
    u.country,
    u.institute,
    u.city,
    tt.test_id,
    t.name AS test_name,
    tt.created_at,
    tt.finished_at,
    t.duration AS time_limit,
    t.pass_mark,
    t.question_limit,
    tt.no_of_questions,
    ar.answer_rows,
    ar.attempted_questions,
    ar.correct_answers,
    ar.wrong_answers,
    tt.marks AS marks,
    ar.answer_grade_sum_diagnostic,
    qr.total_questions,
    qr.total_marks,
    tcr.class_id,
    tcr.group_id,
    tcr.test_class_count
FROM {test_takers} tt
LEFT JOIN {tests} t
    ON t.test_id = tt.test_id
LEFT JOIN {users} u
    ON u.user_id = tt.user_id
LEFT JOIN answer_rollup ar
    ON ar.test_taker_id = tt.test_taker_id
LEFT JOIN question_rollup qr
    ON qr.test_id = tt.test_id
LEFT JOIN test_class_rollup tcr
    ON tcr.test_id = tt.test_id
WHERE (:date_start IS NULL OR tt.created_at >= :date_start)
  AND (:date_end IS NULL OR tt.created_at < :date_end)
""".strip()

def render_sql(sql_template, tables):
    required = sorted(set(re.findall(r"\\{([a-zA-Z0-9_]+)\\}", sql_template)))
    missing = [name for name in required if not tables.get(name)]
    if missing:
        raise ValueError(f"Missing table mapping for: {missing}")
    return sql_template.format(**tables)

def read_raw_attempts(engine):
    sql = render_sql(RAW_ATTEMPT_SQL, TABLES)
    params = {"date_start": DATE_START, "date_end": DATE_END}
    return pd.read_sql_query(text(sql), engine, params=params)

# raw_attempts = read_raw_attempts(engine)
# display(raw_attempts.head())
# print(raw_attempts.shape)

## Optional Class Membership Enrichment

`test_classes` can tell us which classes are attached to a test. For learner cohort analysis, prefer a real membership table that connects `user_id` or `subscriber_id` to `class_id` over time.

If membership effective dates exist, join the learner to the class active at the attempt `created_at`. If they do not exist, mark the cohort as inferred/current-state only.

In [ ]:
CLASS_MEMBERSHIP_SQL = """
-- Replace this with the confirmed membership table and columns.
-- Expected output columns:
-- user_id, subscriber_id, class_id, membership_created_at, membership_updated_at, membership_active
SELECT
    user_id,
    subscriber_id,
    class_id,
    created_at AS membership_created_at,
    updated_at AS membership_updated_at,
    active AS membership_active
FROM {class_memberships}
""".strip()

def attach_class_membership(raw_attempts, membership_df=None):
    df = raw_attempts.copy()
    df["class_source"] = np.where(df.get("class_id").notna(), "test_classes", "missing")
    if membership_df is None or membership_df.empty:
        df["cohort_join_status"] = "test_class_only_no_membership_table"
        return df

    # Conservative default: exact user/subscriber/class match. Tighten with effective dates when confirmed.
    keys = [key for key in ["user_id", "subscriber_id", "class_id"] if key in df.columns and key in membership_df.columns]
    enriched = df.merge(membership_df, on=keys, how="left", suffixes=("", "_membership"))
    enriched["cohort_join_status"] = np.where(
        enriched.get("membership_created_at").notna(),
        "membership_matched",
        "membership_not_matched",
    )
    enriched["class_source"] = np.where(
        enriched["cohort_join_status"].eq("membership_matched"),
        "class_membership",
        enriched["class_source"],
    )
    return enriched

# membership_df = pd.read_sql_query(text(render_sql(CLASS_MEMBERSHIP_SQL, TABLES)), engine)
# raw_attempts = attach_class_membership(raw_attempts, membership_df)

## DQ Annotation Layer

This layer annotates rows before producing filtered outputs. It preserves repeated attempts for proxy sequence analysis and only removes exact duplicate rows if requested.

In [ ]:
def numeric_col(df, column_name):
    if column_name not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[column_name], errors="coerce")

def canonical_denominator_with_source(df):
    marks = numeric_col(df, "marks")
    max_marks = numeric_col(df, "max_marks_effective")
    no_of_questions = numeric_col(df, "no_of_questions")
    total_questions = numeric_col(df, "total_questions")
    question_limit = numeric_col(df, "question_limit")
    total_marks = numeric_col(df, "total_marks")

    delivered_question_count = no_of_questions.where(no_of_questions.gt(0)).fillna(question_limit.where(question_limit.gt(0))).fillna(total_questions.where(total_questions.gt(0)))
    total_questions_suspect = delivered_question_count.gt(0) & total_questions.gt(delivered_question_count * 5)
    total_marks_suspect = total_marks.gt(0) & delivered_question_count.gt(0) & total_marks.gt(delivered_question_count * 5)
    max_marks_suspect = max_marks.gt(0) & delivered_question_count.gt(0) & max_marks.gt(delivered_question_count * 5)
    max_marks_usable = max_marks.gt(0) & ~max_marks_suspect & (marks.isna() | marks.le(max_marks))
    total_marks_usable = total_marks.gt(0) & ~total_marks_suspect & (marks.isna() | marks.le(total_marks))

    # In this export, test_takers.marks tracks correct-answer marks/counts. Prefer
    # test_takers.no_of_questions, which is the delivered attempt count. tests.total_questions
    # can be the full randomized question bank and is only a last-resort fallback.
    denom = pd.Series(np.nan, index=df.index)
    source = pd.Series("missing", index=df.index, dtype="object")
    denom = denom.where(~max_marks_usable, max_marks)
    source = source.where(~max_marks_usable, "max_marks_effective")
    use_no_of_questions = denom.isna() & no_of_questions.gt(0)
    denom = denom.where(~use_no_of_questions, no_of_questions)
    source = source.where(~use_no_of_questions, "no_of_questions")
    use_question_limit = denom.isna() & question_limit.gt(0)
    denom = denom.where(~use_question_limit, question_limit)
    source = source.where(~use_question_limit, "question_limit")
    use_total_questions = denom.isna() & total_questions.gt(0)
    denom = denom.where(~use_total_questions, total_questions)
    source = source.where(~use_total_questions, "total_questions")
    use_total_marks = denom.isna() & total_marks_usable
    denom = denom.where(~use_total_marks, total_marks)
    source = source.where(~use_total_marks, "total_marks")
    return denom.where(denom.gt(0)), source, total_marks_suspect.fillna(False), total_questions_suspect.fillna(False)

def add_dq_flags(raw_df):
    df = raw_df.copy()
    for col in ["created_at", "finished_at"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df["attempted_questions_raw"] = numeric_col(df, "attempted_questions")
    df["correct_answers"] = numeric_col(df, "correct_answers")
    df["wrong_answers"] = numeric_col(df, "wrong_answers")
    df["marks"] = numeric_col(df, "marks")
    df["accuracy_denominator"], df["accuracy_denominator_source"], df["total_marks_suspect"], df["total_questions_suspect"] = canonical_denominator_with_source(df)
    df["avg_accuracy_safe"] = (df["marks"] / df["accuracy_denominator"]).clip(lower=0, upper=1)

    finished_exists = "finished_at" in raw_df.columns
    activity_evidence = (
        df["attempted_questions_raw"].fillna(0).gt(0)
        | df["correct_answers"].fillna(0).gt(0)
        | df["wrong_answers"].fillna(0).gt(0)
        | df["marks"].fillna(0).gt(0)
    )
    if finished_exists:
        df["missing_finished_at_column"] = False
        df["completion_source"] = "finished_at"
        df["completion_status"] = np.where(df["finished_at"].notna(), "verified_complete", "incomplete")
    else:
        df["missing_finished_at_column"] = True
        df["completion_source"] = np.where(activity_evidence, "fallback_activity_evidence", "missing_finished_at_column")
        df["completion_status"] = np.where(activity_evidence, "unknown_but_usable", "incomplete")

    df["inactive_zero_attempt"] = df["attempted_questions_raw"].fillna(0).eq(0)
    df["missing_question_support"] = df["accuracy_denominator"].isna()
    df["missing_pass_mark"] = numeric_col(df, "pass_mark").isna()
    df["exact_duplicate_row"] = df.duplicated(keep=False)
    df["dq_eligible_proxy_sequence"] = (
        df["completion_status"].isin(["verified_complete", "unknown_but_usable"])
        & df["accuracy_denominator"].notna()
        & ~df["inactive_zero_attempt"]
    )
    df["dq_eligible_published"] = df["dq_eligible_proxy_sequence"] & ~df["exact_duplicate_row"]
    return df

# dq_attempts = add_dq_flags(raw_attempts)
# display(dq_attempts.head())

## Published KPI Dataset

Published KPI rows may use best-attempt dedupe. This output is for rankings, institution rollups, and published summaries. It is not used for BLS/ALS/CAS sequence calculations.

In [ ]:
def build_published_kpi_dataset(dq_attempts):
    df = dq_attempts.loc[dq_attempts["dq_eligible_published"]].copy()
    sort_cols = [col for col in ["user_id", "test_id", "avg_accuracy_safe", "marks", "created_at"] if col in df.columns]
    ascending = [True, True, False, False, False][: len(sort_cols)]
    df = df.sort_values(sort_cols, ascending=ascending)
    return df.drop_duplicates(["user_id", "test_id"], keep="first")

# published_kpi = build_published_kpi_dataset(dq_attempts)
# print("published_kpi", published_kpi.shape)
# published_kpi.to_csv(EXPORT_DIR / "published_kpi_user_test.csv", index=False)

## Proxy-Sequence Dataset

Proxy sequence rows must preserve repeated eligible attempts. This output supports inferred BLS proxy, current ALS proxy, potential ALS proxy, learning gain proxy, and CAS proxy.

In [ ]:
def build_proxy_sequence_dataset(dq_attempts):
    df = dq_attempts.loc[dq_attempts["dq_eligible_proxy_sequence"]].copy()
    return df.sort_values(["user_id", "test_id", "created_at", "attempt_id"], na_position="last")

def build_user_test_readiness_summary(proxy_sequence, published_kpi=None):
    rows = []
    group_cols = ["user_id", "test_id"]
    for (user_id, test_id), group in proxy_sequence.groupby(group_cols, dropna=False):
        attempts = group.sort_values(["created_at", "attempt_id"], na_position="last")
        first = attempts.iloc[0]
        later = attempts.iloc[1:]
        current = later.iloc[-1] if len(later) else pd.Series(dtype="object")
        potential = later.sort_values("avg_accuracy_safe", ascending=False).iloc[0] if len(later) else pd.Series(dtype="object")

        bls = first.get("avg_accuracy_safe", np.nan)
        current_als = current.get("avg_accuracy_safe", np.nan) if len(later) else np.nan
        potential_als = potential.get("avg_accuracy_safe", np.nan) if len(later) else np.nan
        bls_row_number = int(attempts.index.get_loc(first.name)) + 1
        current_als_row_number = int(attempts.index.get_loc(current.name)) + 1 if len(later) else np.nan
        potential_als_row_number = int(attempts.index.get_loc(potential.name)) + 1 if len(later) else np.nan

        rows.append({
            "user_id": user_id,
            "test_id": test_id,
            "test_name": first.get("test_name"),
            "subscriber_id": first.get("subscriber_id"),
            "first_name": first.get("first_name"),
            "last_name": first.get("last_name"),
            "learner_id": first.get("learner_id"),
            "country": first.get("country"),
            "institute": first.get("institute"),
            "city": first.get("city"),
            "class_id": first.get("class_id"),
            "group_id": first.get("group_id"),
            "attempt_count": len(attempts),
            "bls_attempt_id": first.get("attempt_id"),
            "bls_row_number": bls_row_number,
            "bls_created_at": first.get("created_at"),
            "bls_score_raw": first.get("marks"),
            "bls_score_denominator": first.get("accuracy_denominator"),
            "bls_score_denominator_source": first.get("accuracy_denominator_source"),
            "bls_total_marks_suspect": first.get("total_marks_suspect"),
            "bls_total_questions_suspect": first.get("total_questions_suspect"),
            "bls_score_pct": bls * 100 if pd.notna(bls) else np.nan,
            "current_als_attempt_id": current.get("attempt_id") if len(later) else np.nan,
            "current_als_row_number": current_als_row_number,
            "current_als_created_at": current.get("created_at") if len(later) else pd.NaT,
            "current_als_score_raw": current.get("marks") if len(later) else np.nan,
            "current_als_score_denominator": current.get("accuracy_denominator") if len(later) else np.nan,
            "current_als_score_denominator_source": current.get("accuracy_denominator_source") if len(later) else np.nan,
            "current_als_total_marks_suspect": current.get("total_marks_suspect") if len(later) else np.nan,
            "current_als_total_questions_suspect": current.get("total_questions_suspect") if len(later) else np.nan,
            "current_als_score_pct": current_als * 100 if pd.notna(current_als) else np.nan,
            "potential_als_attempt_id": potential.get("attempt_id") if len(later) else np.nan,
            "potential_als_row_number": potential_als_row_number,
            "potential_als_created_at": potential.get("created_at") if len(later) else pd.NaT,
            "potential_als_score_raw": potential.get("marks") if len(later) else np.nan,
            "potential_als_score_denominator": potential.get("accuracy_denominator") if len(later) else np.nan,
            "potential_als_score_denominator_source": potential.get("accuracy_denominator_source") if len(later) else np.nan,
            "potential_als_total_marks_suspect": potential.get("total_marks_suspect") if len(later) else np.nan,
            "potential_als_total_questions_suspect": potential.get("total_questions_suspect") if len(later) else np.nan,
            "potential_als_score_pct": potential_als * 100 if pd.notna(potential_als) else np.nan,
            "learning_gain_pct": (current_als - bls) * 100 if pd.notna(current_als) and pd.notna(bls) else np.nan,
            "potential_gain_pct": (potential_als - bls) * 100 if pd.notna(potential_als) and pd.notna(bls) else np.nan,
            "proxy_evidence_band": "high" if len(attempts) >= 3 else ("medium" if len(attempts) == 2 else "low"),
            "learn_smarter_mapping_status": "test_exercise_proxy_no_topic_id",
        })
    summary = pd.DataFrame(rows)
    if published_kpi is not None and not published_kpi.empty:
        published_cols = ["user_id", "test_id", "avg_accuracy_safe", "marks"]
        available = [col for col in published_cols if col in published_kpi.columns]
        summary = summary.merge(
            published_kpi[available].rename(columns={"avg_accuracy_safe": "published_accuracy_safe", "marks": "published_marks"}),
            on=["user_id", "test_id"],
            how="left",
        )
    return summary

# proxy_sequence = build_proxy_sequence_dataset(dq_attempts)
# user_test_summary = build_user_test_readiness_summary(proxy_sequence, published_kpi)
# print("proxy_sequence", proxy_sequence.shape)
# print("user_test_summary", user_test_summary.shape)
# proxy_sequence.to_csv(EXPORT_DIR / "proxy_sequence_attempts.csv", index=False)
# user_test_summary.to_csv(EXPORT_DIR / "v13_user_test_readiness_summary.csv", index=False)

## Group / Cohort Summary

CAS remains a proxy in v1.3. If true class membership is not confirmed, label cohort outputs as inferred.

In [ ]:
def build_group_readiness_summary(user_test_summary):
    group_frames = []
    for group_level, group_col in [("class", "class_id"), ("institute", "institute"), ("city", "city"), ("country", "country"), ("institution_or_subscriber", "subscriber_id"), ("test", "test_id")]:
        if group_col not in user_test_summary.columns:
            continue
        # Avoid grouping by test_id twice for the test-level summary.
        group_keys = [group_col] if group_col == "test_id" else [group_col, "test_id"]
        grouped = user_test_summary.groupby(group_keys, dropna=False).agg(
            learner_count=("user_id", "nunique"),
            repeated_group_count=("attempt_count", lambda s: int((s >= 2).sum())),
            mean_bls_score_pct=("bls_score_pct", "mean"),
            mean_current_als_score_pct=("current_als_score_pct", "mean"),
            mean_potential_als_score_pct=("potential_als_score_pct", "mean"),
            mean_learning_gain_pct=("learning_gain_pct", "mean"),
        ).reset_index()
        if group_col == "test_id":
            grouped["group_id"] = grouped["test_id"]
        else:
            grouped = grouped.rename(columns={group_col: "group_id"})
        grouped["group_level"] = group_level
        grouped["cas_proxy_score_pct"] = grouped["mean_current_als_score_pct"]
        grouped["caveat_note"] = "CAS proxy from test/exercise attempts; not true class ALS calibration."
        group_frames.append(grouped)
    return pd.concat(group_frames, ignore_index=True) if group_frames else pd.DataFrame()

# group_summary = build_group_readiness_summary(user_test_summary)
# display(group_summary.head())
# group_summary.to_csv(EXPORT_DIR / "v13_group_readiness_summary.csv", index=False)

## Smoke Checks

These checks should run before any export is treated as dashboard-ready.

In [ ]:
def smoke_report(raw_attempts, dq_attempts, published_kpi, proxy_sequence, user_test_summary):
    repeated_groups = proxy_sequence.groupby(["user_id", "test_id"], dropna=False).size()
    report = {
        "raw_rows": len(raw_attempts),
        "dq_annotated_rows": len(dq_attempts),
        "published_kpi_rows": len(published_kpi),
        "proxy_sequence_rows": len(proxy_sequence),
        "user_test_summary_rows": len(user_test_summary),
        "repeated_user_test_groups": int((repeated_groups >= 2).sum()),
        "inactive_zero_attempt_rows": int(dq_attempts.get("inactive_zero_attempt", pd.Series(dtype=bool)).sum()),
        "total_marks_suspect_rows": int(dq_attempts.get("total_marks_suspect", pd.Series(dtype=bool)).sum()),
        "total_questions_suspect_rows": int(dq_attempts.get("total_questions_suspect", pd.Series(dtype=bool)).sum()),
        "missing_finished_at_column": bool(dq_attempts.get("missing_finished_at_column", pd.Series([False])).any()),
        "accuracy_min": float(dq_attempts["avg_accuracy_safe"].min(skipna=True)) if "avg_accuracy_safe" in dq_attempts else None,
        "accuracy_max": float(dq_attempts["avg_accuracy_safe"].max(skipna=True)) if "avg_accuracy_safe" in dq_attempts else None,
        "learning_gain_min": float(user_test_summary["learning_gain_pct"].min(skipna=True)) if "learning_gain_pct" in user_test_summary else None,
        "learning_gain_max": float(user_test_summary["learning_gain_pct"].max(skipna=True)) if "learning_gain_pct" in user_test_summary else None,
    }
    return pd.DataFrame([report])

# report = smoke_report(raw_attempts, dq_attempts, published_kpi, proxy_sequence, user_test_summary)
# display(report.T)
# report.to_csv(EXPORT_DIR / "smoke_report.csv", index=False)

## Export Manifest

Keep a small manifest with row counts, run date, and caveats beside the exported CSVs.

In [ ]:
def write_manifest(export_dir, report_df, extra_notes=None):
    manifest = {
        "run_label": RUN_LABEL,
        "generated_at_utc": pd.Timestamp.utcnow().isoformat(),
        "v1_3_scope": "Test / Exercise Readiness proxy dataset",
        "topic_level_status": "Deferred to v2; no canonical topic_id in current source.",
        "subject_level_status": "May be inferred from test_name only; not canonical.",
        "class_level_status": "Use class_id with membership validation when available.",
        "dq_note": "Raw rows are annotated before filtering. Published KPI and proxy-sequence outputs are separate.",
        "smoke_report": report_df.to_dict(orient="records")[0] if report_df is not None and not report_df.empty else {},
        "extra_notes": extra_notes or [],
    }
    path = Path(export_dir) / "manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return path

# manifest_path = write_manifest(EXPORT_DIR, report)
# print(manifest_path)